# Paper 8 backbone on YOLO26n: CDA (C2f_DWS_Add) + SimSPPF

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 8:** `C3k2` at layers 2, 4, 6, 8 replaced by **CDA**; variant 2 also replaces the `SPPF` at layer 9 with **SimSPPF** |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

Two variants are trained:

| Run | Backbone |
|---|---|
| `bb_cda` | CDA at layers 2, 4, 6, 8 |
| `bb_cda_simsppf` | CDA at layers 2, 4, 6, 8 + SimSPPF at layer 9 |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so cda_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 8 backbone: CDA and SimSPPF

**What CDA does** (Paper 8, Section 3.2.1).
1. A 1×1 conv (`cv1`) gives 2c channels, split into **x1** and **x2**.
2. For each block: **x2 = DWS(x2) + x2**, where DWS = depthwise 3×3 conv + pointwise 1×1 conv (each with BN + SiLU).
3. Only x1 and the updated x2 are concatenated (2c channels) and a 1×1 conv (`cv2`) gives the output.

There is no growing concatenation, so the channel count stays constant inside the block, and the full 3×3 convs of the
base bottleneck become depthwise separable convs.

**What SimSPPF does** (Paper 8, Section 3.2.2). The same structure as SPPF (k=5, 3 poolings, same shortcut),
but the activation of both 1×1 convs is **ReLU** (the original SimSPPF of YOLOv6).

**Where they sit in YOLO26n.**

| Layer | Base YOLO26n | `bb_cda` | `bb_cda_simsppf` | Output |
|---|---|---|---|---|
| 2 | C3k2 | **CDA** | **CDA** | P2/4 (160 × 160) |
| 4 | C3k2 | **CDA** | **CDA** | P3/8 (80 × 80) |
| 6 | C3k2 (c3k) | **CDA** | **CDA** | P4/16 (40 × 40) |
| 8 | C3k2 (c3k) | **CDA** | **CDA** | P5/32 (20 × 20) |
| 9 | SPPF (SiLU) | SPPF | **SimSPPF** (ReLU) | P5/32 (20 × 20) |

Input and output channels stay the same, so no layer index changes and the neck connections stay the same.
The neck C3k2 blocks (layers 13, 16, 19, 22) are not changed: only the backbone is taken from Paper 8.

**Pretrained weights.**
- CDA: `cv1` is copied from the base C3k2 (same shape). The base `cv2` reads [x1, x2, bottleneck output] (3c channels);
  CDA's `cv2` reads [x1, x2] (2c), so the first 2c input columns of the base `cv2` and its BatchNorm are copied.
  Only the DWS convs start from scratch.
- SimSPPF: every weight is copied from the base SPPF; only the activation changes.

**Assumptions.** Paper 8 does not say which C3k2 blocks become CDA; here only the 4 backbone ones are changed.
Copying the pretrained `cv1`/`cv2` weights is added in this notebook (Paper 8 does not describe it).

In [ ]:
%%writefile cda_backbone.py
"""Paper 8 backbone for YOLO26n: C3k2 at layers 2, 4, 6, 8 -> CDA (C2f_DWS_Add); optional SPPF at layer 9 -> SimSPPF (ReLU)."""
from copy import deepcopy

import torch
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import SPPF, C3k2
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

CDA_LAYERS = (2, 4, 6, 8)
SIMSPPF_LAYER = 9


class DWS(nn.Module):
    """Depthwise separable convolution: depthwise 3x3 + pointwise 1x1, each with BN + SiLU."""

    def __init__(self, c):
        super().__init__()
        self.dw = Conv(c, c, 3, 1, g=c)
        self.pw = Conv(c, c, 1, 1)

    def forward(self, x):
        return self.pw(self.dw(x))


class CDA(nn.Module):
    """C2f_DWS_Add: split into x1, x2; x2 = DWS(x2) + x2 (n times); concat [x1, x2]; 1x1 conv."""

    def __init__(self, c1, c2, n=1, c=None):
        super().__init__()
        self.c = c or c2 // 2  # hidden channels
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.m = nn.ModuleList(DWS(self.c) for _ in range(n))
        self.cv2 = Conv(2 * self.c, c2, 1, 1)

    def forward(self, x):
        x1, x2 = self.cv1(x).chunk(2, 1)
        for m in self.m:
            x2 = m(x2) + x2
        return self.cv2(torch.cat((x1, x2), 1))


class SimSPPF(SPPF):
    """SPPF with ReLU activations (YOLOv6 SimSPPF). Same weights and forward as SPPF."""


def _swap(model, i, new, name):
    old = model.model[i]
    new = new.to(next(old.parameters()).device)
    new.i, new.f, new.type = old.i, old.f, name  # keep the routing attributes used by the model's forward
    new.np = sum(p.numel() for p in new.parameters())
    model.model[i] = new


def add_cda(model, layers=CDA_LAYERS):
    """Replace C3k2 at the given layers with CDA, copying cv1 and the matching part of cv2. Returns the model."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, C3k2), f"layer {i} is {type(old).__name__}, not C3k2"
        c, n = old.c, len(old.m)
        new = CDA(old.cv1.conv.in_channels, old.cv2.conv.out_channels, n=n, c=c)
        new.cv1.load_state_dict(old.cv1.state_dict())
        with torch.no_grad():  # base cv2 reads [x1, x2, y1..yn]; CDA cv2 reads [x1, x2]
            new.cv2.conv.weight.copy_(old.cv2.conv.weight[:, : 2 * c])
        new.cv2.bn.load_state_dict(old.cv2.bn.state_dict())
        _swap(model, i, new, "CDA")
    return model


def add_simsppf(model, layer=SIMSPPF_LAYER):
    """Turn the SPPF at the given layer into SimSPPF (all weights kept, activations -> ReLU). Returns the model."""
    old = model.model[layer]
    assert type(old) is SPPF, f"layer {layer} is {type(old).__name__}, not SPPF"
    new = deepcopy(old)
    new.__class__ = SimSPPF
    new.cv1.act = nn.ReLU()
    new.cv2.act = nn.ReLU()
    _swap(model, layer, new, "SimSPPF")
    return model


class CDATrainer(DetectionTrainer):
    """Loads the pretrained weights first, then swaps C3k2 -> CDA (and SPPF -> SimSPPF when SIMSPPF is True)."""

    SIMSPPF = False

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_cda(model)
        if self.SIMSPPF:
            add_simsppf(model)
        if RANK in {-1, 0}:
            print(f"Paper 8 backbone: CDA at layers {CDA_LAYERS}" + (f", SimSPPF at layer {SIMSPPF_LAYER}" if self.SIMSPPF else ""))
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


class CDASimSPPFTrainer(CDATrainer):
    SIMSPPF = True


In [ ]:
import importlib, cda_backbone
importlib.reload(cda_backbone)
from cda_backbone import add_cda, add_simsppf, CDATrainer, CDASimSPPFTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs the two Paper 8 backbones
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
cda = add_cda(deepcopy(base)).eval()
cda_sim = add_simsppf(add_cda(deepcopy(base))).eval()
with torch.no_grad():
    for m in (cda, cda_sim):
        _ = m(torch.zeros(1, 3, 640, 640))  # forward check
print("layers 2-9 (CDA + SimSPPF):", [type(cda_sim.model[i]).__name__ for i in range(2, 10)])
print("layer 9 activations:", type(cda_sim.model[9].cv1.act).__name__, type(cda_sim.model[9].cv2.act).__name__)
for tag, m in (("base", base), ("CDA", cda), ("CDA + SimSPPF", cda_sim)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:14s} Params {pm:.3f} M | GFLOPs {gf:.2f}")

## 4. Training

Two Paper 8 backbone variants are trained with identical settings.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name, trainer)
VARIANTS = [
    ("CDA Backbone (Paper 8)",           "bb_cda",         CDATrainer),
    ("CDA + SimSPPF Backbone (Paper 8)", "bb_cda_simsppf", CDASimSPPFTrainer),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run, trainer in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=trainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [ ]:
import cda_backbone  # must be imported before loading a CDA / SimSPPF checkpoint

rows = []
for label, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper8_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_cda_ncnn_model/`, `bb_cda_simsppf_ncnn_model/` (NCNN, imgsz 640) | Raspberry Pi 5 |
| `bb_cda.pt`, `bb_cda_simsppf.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `cda_backbone.py` | Jetson (needed to load the CDA / SimSPPF `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper8_backbone_edge_bench.zip paper8_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper8_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("cda_backbone.py", EDGE / "cda_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import cda_backbone  # needed to load the CDA / SimSPPF checkpoints
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper8_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper8_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the two Paper 8 backbone models
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_cda": None, "bb_cda_simsppf": None}
FPS_JETSON = {"bb_cda": None, "bb_cda_simsppf": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 8. Zip all results

Run this last. It creates **`paper8_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper8_backbone_edge_bench/` | NCNN + `.pt` models, 50 bench images, `cda_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `bb_cda_best.pt`, `bb_cda_simsppf_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper8_backbone_all_results.zip"
folders = [WORK / "paper8_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run, _ in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")